# XGBoost Baseline - Optimized (Google Colab Version)

**SOC Analyst Triage AI Assistant**

**⚠️ IMPORTANT:** This notebook is designed for Google Colab.

**Prerequisites:**
1. Run `02_preprocessing.ipynb` locally to generate processed CSVs
2. Upload processed CSVs to Google Drive: `SOC_Project/data/processed/`
3. Run this notebook in Google Colab with GPU enabled

**Authors:** Napo Qheku & Kabelo Thesele  
**Supervisor:** Mr. Lekuba Ntoane  
**Date:** February 2026

---

## 🔧 Setup Instructions

### Step 1: Upload Data to Google Drive

Create this folder structure in your Google Drive:
```
My Drive/
└── SOC_Project/
    ├── data/
    │   └── processed/
    │       ├── X_train.csv
    │       ├── X_test.csv
    │       ├── y_train.csv
    │       └── y_test.csv
    └── models/  (will be created automatically)
```

### Step 2: Enable GPU Runtime

1. Click **Runtime** → **Change runtime type**
2. Select **T4 GPU** under Hardware accelerator
3. Click **Save**

### Step 3: Run All Cells

Click **Runtime** → **Run all** or run cells one by one with Shift+Enter

## 1. Mount Google Drive and Install Packages

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted successfully!")
print("   Your files are accessible at: /content/drive/MyDrive/")

In [ ]:
# Install XGBoost if not already installed
!pip install xgboost -q

print("✅ XGBoost installed")

## 2. Setup and Configuration

In [ ]:
# Core libraries
import numpy as np
import pandas as pd
from pathlib import Path
import warnings
import json
import os
warnings.filterwarnings('ignore')

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

# Machine Learning
from sklearn.model_selection import (
    train_test_split, RandomizedSearchCV, 
    cross_val_score, StratifiedKFold
)
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, 
    roc_auc_score, roc_curve, precision_recall_curve
)
from xgboost import XGBClassifier
import xgboost as xgb

# Configuration
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Google Drive paths
DRIVE_BASE = Path("/content/drive/MyDrive/SOC_Project")
PROCESSED_DIR = DRIVE_BASE / "data/processed"
MODEL_DIR = DRIVE_BASE / "models"

# Create models directory
MODEL_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 60)
print("SETUP COMPLETE")
print("=" * 60)
print(f"✅ Random seed: {RANDOM_SEED}")
print(f"✅ XGBoost version: {xgb.__version__}")
print(f"✅ NumPy version: {np.__version__}")
print(f"✅ Pandas version: {pd.__version__}")
print(f"\n📁 Paths:")
print(f"   Data: {PROCESSED_DIR}")
print(f"   Models: {MODEL_DIR}")
print("=" * 60)

## 3. Verify Data Files

In [ ]:
print("=" * 60)
print("VERIFYING DATA FILES")
print("=" * 60)

# Check if folder exists
if not PROCESSED_DIR.exists():
    print(f"\n❌ ERROR: Folder not found!")
    print(f"   Expected: {PROCESSED_DIR}")
    print("\n📝 Action Required:")
    print("   1. Create folder in Google Drive: SOC_Project/data/processed/")
    print("   2. Upload these files:")
    print("      - X_train.csv")
    print("      - X_test.csv")
    print("      - y_train.csv")
    print("      - y_test.csv")
    print("   3. Refresh this cell")
    raise FileNotFoundError(f"Folder not found: {PROCESSED_DIR}")

# Check required files
required_files = ['X_train.csv', 'X_test.csv', 'y_train.csv', 'y_test.csv']
found_files = []
missing_files = []

for file in required_files:
    filepath = PROCESSED_DIR / file
    if filepath.exists():
        size_mb = filepath.stat().st_size / (1024 * 1024)
        found_files.append((file, size_mb))
        print(f"   ✅ {file:<15} ({size_mb:>6.2f} MB)")
    else:
        missing_files.append(file)
        print(f"   ❌ {file:<15} MISSING!")

if missing_files:
    print(f"\n❌ ERROR: Missing {len(missing_files)} file(s)!")
    print(f"   Missing: {missing_files}")
    print("\n📝 Upload missing files to Google Drive and try again.")
    raise FileNotFoundError(f"Missing files: {missing_files}")

print("\n✅ All required files found!")
print("=" * 60)

## 4. Load Preprocessed Data

In [ ]:
print("=" * 60)
print("LOADING PREPROCESSED DATA")
print("=" * 60)

print("\n⏳ Loading data from Google Drive...")
print("   (This may take 1-2 minutes for large files)\n")

# Load data
X_train_full = pd.read_csv(PROCESSED_DIR / "X_train.csv")
print("   ✓ X_train.csv loaded")

X_test = pd.read_csv(PROCESSED_DIR / "X_test.csv")
print("   ✓ X_test.csv loaded")

y_train_full = pd.read_csv(PROCESSED_DIR / "y_train.csv").squeeze()
print("   ✓ y_train.csv loaded")

y_test = pd.read_csv(PROCESSED_DIR / "y_test.csv").squeeze()
print("   ✓ y_test.csv loaded")

print("\n✅ All data loaded successfully!")
print(f"\n📊 Dataset Shapes:")
print(f"   Training features: {X_train_full.shape}")
print(f"   Training labels:   {y_train_full.shape}")
print(f"   Test features:     {X_test.shape}")
print(f"   Test labels:       {y_test.shape}")
print(f"   Total features:    {X_train_full.shape[1]}")

# Verify data integrity
print(f"\n🔍 Data Integrity Checks:")
assert X_train_full.shape[0] == 175341, "Unexpected training set size"
print("   ✓ Training set size correct (175,341)")

assert X_test.shape[0] == 82332, "Unexpected test set size"
print("   ✓ Test set size correct (82,332)")

assert X_train_full.shape[1] == X_test.shape[1], "Feature dimension mismatch"
print(f"   ✓ Feature dimensions match ({X_train_full.shape[1]} features)")

assert X_train_full.isnull().sum().sum() == 0, "Missing values in training data"
assert X_test.isnull().sum().sum() == 0, "Missing values in test data"
print("   ✓ No missing values")

print(f"\n📈 Class Distribution:")
train_attack_pct = y_train_full.sum()/len(y_train_full)*100
test_attack_pct = y_test.sum()/len(y_test)*100
print(f"   Training: {train_attack_pct:.2f}% malicious, {100-train_attack_pct:.2f}% benign")
print(f"   Testing:  {test_attack_pct:.2f}% malicious, {100-test_attack_pct:.2f}% benign")

print("\n" + "=" * 60)

## 5. Train-Validation Split

In [ ]:
print("=" * 60)
print("TRAIN-VALIDATION SPLIT")
print("=" * 60)

# Split training data into train and validation (80/20)
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.2,
    stratify=y_train_full,
    random_state=RANDOM_SEED
)

print("\n📊 Dataset Splits:")
print(f"   Training:   {X_train.shape[0]:>7,} samples ({X_train.shape[0]/len(X_train_full)*100:>5.1f}%)")
print(f"   Validation: {X_val.shape[0]:>7,} samples ({X_val.shape[0]/len(X_train_full)*100:>5.1f}%)")
print(f"   Testing:    {X_test.shape[0]:>7,} samples (held-out)")
print(f"   Features:   {X_train.shape[1]:>7,}")

print("\n✓ Stratification Verified:")
print(f"   Training:   {y_train.sum()/len(y_train)*100:.2f}% malicious")
print(f"   Validation: {y_val.sum()/len(y_val)*100:.2f}% malicious")
print(f"   Testing:    {y_test.sum()/len(y_test)*100:.2f}% malicious")

print("\n📝 Validation Set Purpose:")
print("   • Hyperparameter optimization")
print("   • Learning curve monitoring")
print("   • Early stopping")
print("   • Threshold optimization")
print("\n   ⚠️  Test set: Only for final evaluation!")
print("=" * 60)

## 6. Baseline Model (Default Hyperparameters)

In [ ]:
print("=" * 60)
print("BASELINE MODEL (DEFAULT HYPERPARAMETERS)")
print("=" * 60)

baseline_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    objective='binary:logistic',
    random_state=RANDOM_SEED,
    n_jobs=-1,
    tree_method='hist'  # Faster on CPU
)

print("\n⏳ Training baseline model...")
baseline_model.fit(X_train, y_train, verbose=False)
print("✅ Training complete!")

# Evaluate
y_val_pred_baseline = baseline_model.predict(X_val)
y_val_proba_baseline = baseline_model.predict_proba(X_val)[:, 1]

baseline_metrics = {
    'Accuracy': accuracy_score(y_val, y_val_pred_baseline),
    'Precision': precision_score(y_val, y_val_pred_baseline),
    'Recall': recall_score(y_val, y_val_pred_baseline),
    'F1-Score': f1_score(y_val, y_val_pred_baseline),
    'ROC-AUC': roc_auc_score(y_val, y_val_proba_baseline)
}

print("\n📊 Baseline Performance (Validation):")
for metric, value in baseline_metrics.items():
    print(f"   {metric:12} : {value:.4f}")

print("\n📝 This is our starting point before optimization")
print("=" * 60)

## 7. Hyperparameter Optimization

**Note:** This will take 15-20 minutes. Grab a coffee! ☕

In [ ]:
print("=" * 60)
print("HYPERPARAMETER OPTIMIZATION")
print("=" * 60)

# Define search space
param_distributions = {
    'n_estimators': [100, 200, 300],
    'max_depth': [4, 6, 8],
    'learning_rate': [0.05, 0.1, 0.2],
    'subsample': [0.7, 0.8, 0.9],
    'colsample_bytree': [0.7, 0.8, 0.9],
    'min_child_weight': [1, 3, 5],
    'gamma': [0, 0.1, 0.2]
}

print("\n🔍 Search Space:")
for param, values in param_distributions.items():
    print(f"   {param:20} : {values}")

total_combinations = np.prod([len(v) for v in param_distributions.values()])
print(f"\n   Total combinations: {total_combinations:,}")
print(f"   Testing: 50 random combinations")

# RandomizedSearchCV
random_search = RandomizedSearchCV(
    estimator=XGBClassifier(
        objective='binary:logistic',
        random_state=RANDOM_SEED,
        n_jobs=-1,
        tree_method='hist'
    ),
    param_distributions=param_distributions,
    n_iter=50,
    cv=3,
    scoring='f1',
    n_jobs=-1,
    verbose=2,
    random_state=RANDOM_SEED,
    return_train_score=True
)

print("\n⚙️  Starting hyperparameter search...")
print("   (Estimated time: 15-20 minutes)")
print("   You can minimize this tab and check back later.\n")

random_search.fit(X_train, y_train)

print("\n" + "=" * 60)
print("✅ HYPERPARAMETER OPTIMIZATION COMPLETE!")
print("=" * 60)

In [ ]:
# Display results
print("🏆 Best Hyperparameters Found:")
for param, value in random_search.best_params_.items():
    print(f"   {param:20} : {value}")

print(f"\n📊 Best Cross-Validation F1-Score: {random_search.best_score_:.4f}")

# Get best model and evaluate
best_model = random_search.best_estimator_

y_val_pred_opt = best_model.predict(X_val)
y_val_proba_opt = best_model.predict_proba(X_val)[:, 1]

optimized_metrics = {
    'Accuracy': accuracy_score(y_val, y_val_pred_opt),
    'Precision': precision_score(y_val, y_val_pred_opt),
    'Recall': recall_score(y_val, y_val_pred_opt),
    'F1-Score': f1_score(y_val, y_val_pred_opt),
    'ROC-AUC': roc_auc_score(y_val, y_val_proba_opt)
}

print("\n📊 Optimized Model Performance (Validation):")
for metric, value in optimized_metrics.items():
    print(f"   {metric:12} : {value:.4f}")

# Comparison
comparison_df = pd.DataFrame({
    'Baseline': baseline_metrics,
    'Optimized': optimized_metrics,
    'Improvement': {k: optimized_metrics[k] - baseline_metrics[k] 
                   for k in baseline_metrics.keys()}
})

print("\n" + "=" * 60)
print("BASELINE vs OPTIMIZED")
print("=" * 60)
print(comparison_df.to_string())
print("\n✓ Positive = Improvement from optimization")
print("=" * 60)

## 8. Cross-Validation

In [ ]:
print("=" * 60)
print("CROSS-VALIDATION (ROBUSTNESS CHECK)")
print("=" * 60)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

print("\n⚙️  Performing 5-fold cross-validation...")
print("   (This will take 5-10 minutes)\n")

cv_scores = {}
for metric in ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']:
    scores = cross_val_score(
        best_model, 
        X_train_full,
        y_train_full,
        cv=cv,
        scoring=metric,
        n_jobs=-1
    )
    cv_scores[metric] = scores

print("📊 Cross-Validation Results:\n")
print(f"{'Metric':<12} {'Mean':>8} {'±Std':>8} {'Min':>8} {'Max':>8}")
print("-" * 48)
for metric, scores in cv_scores.items():
    print(f"{metric.upper():<12} {scores.mean():>8.4f} {scores.std():>8.4f} "
          f"{scores.min():>8.4f} {scores.max():>8.4f}")

print("\n✓ Low std → Stable, robust model")
print("=" * 60)

In [ ]:
# Visualize CV results
cv_df = pd.DataFrame(cv_scores)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cv_df.boxplot(ax=axes[0])
axes[0].set_title('Cross-Validation Score Distribution', fontweight='bold')
axes[0].set_ylabel('Score')
axes[0].set_ylim([0.75, 1.0])
axes[0].grid(True, alpha=0.3)

means = cv_df.mean()
stds = cv_df.std()
axes[1].bar(range(len(means)), means, yerr=stds, capsize=5, alpha=0.7)
axes[1].set_xticks(range(len(means)))
axes[1].set_xticklabels(means.index, rotation=0)
axes[1].set_title('Mean ± Std', fontweight='bold')
axes[1].set_ylabel('Score')
axes[1].set_ylim([0.75, 1.0])
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(MODEL_DIR / 'cross_validation.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved to: models/cross_validation.png")

## 9. Feature Importance

In [ ]:
print("=" * 60)
print("FEATURE IMPORTANCE ANALYSIS")
print("=" * 60)

# Extract feature importance
importance_dict = best_model.get_booster().get_score(importance_type='gain')
feature_names = X_train.columns.tolist()

importance_list = []
for feat_id, importance in importance_dict.items():
    idx = int(feat_id.replace('f', '')) if feat_id.startswith('f') else int(feat_id)
    feat_name = feature_names[idx]
    importance_list.append({'feature': feat_name, 'importance': importance})

importance_df = pd.DataFrame(importance_list).sort_values('importance', ascending=False)

print(f"\n📊 Top 20 Most Important Features:\n")
print(importance_df.head(20).to_string(index=False))

# Save
importance_df.to_csv(MODEL_DIR / 'feature_importance.csv', index=False)
print(f"\n✓ Saved to: models/feature_importance.csv")

In [ ]:
# Visualize
top_n = 20
top_features = importance_df.head(top_n)

plt.figure(figsize=(10, 8))
plt.barh(range(top_n), top_features['importance'].values, color='steelblue')
plt.yticks(range(top_n), top_features['feature'].values, fontsize=10)
plt.xlabel('Importance (Gain)', fontsize=12)
plt.title(f'Top {top_n} Feature Importance', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.savefig(MODEL_DIR / 'feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved to: models/feature_importance.png")

## 10. Learning Curves

In [ ]:
print("=" * 60)
print("LEARNING CURVES (OVERFITTING CHECK)")
print("=" * 60)

print("\n⚙️  Retraining with monitoring...")

final_model = XGBClassifier(**random_search.best_params_)
final_model.set_params(tree_method='hist')  # Ensure hist method

eval_set = [(X_train, y_train), (X_val, y_val)]
final_model.fit(
    X_train, y_train,
    eval_set=eval_set,
    eval_metric=['logloss', 'error'],
    verbose=False
)

results = final_model.evals_result()
print("✅ Training complete")

In [ ]:
# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(len(results['validation_0']['logloss']))

axes[0].plot(epochs, results['validation_0']['logloss'], label='Training', linewidth=2)
axes[0].plot(epochs, results['validation_1']['logloss'], label='Validation', linewidth=2)
axes[0].set_xlabel('Boosting Round')
axes[0].set_ylabel('Log Loss')
axes[0].set_title('Learning Curve: Loss', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs, results['validation_0']['error'], label='Training', linewidth=2)
axes[1].plot(epochs, results['validation_1']['error'], label='Validation', linewidth=2)
axes[1].set_xlabel('Boosting Round')
axes[1].set_ylabel('Error Rate')
axes[1].set_title('Learning Curve: Error', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

# Analyze gap
final_train_loss = results['validation_0']['logloss'][-1]
final_val_loss = results['validation_1']['logloss'][-1]
gap = final_val_loss - final_train_loss

print(f"\n📊 Overfitting Analysis:")
print(f"   Training Loss:   {final_train_loss:.4f}")
print(f"   Validation Loss: {final_val_loss:.4f}")
print(f"   Gap:             {gap:.4f}")

if gap < 0.05:
    print("   ✅ Excellent generalization")
elif gap < 0.10:
    print("   ✅ Good generalization")
else:
    print("   ⚠️  Possible overfitting")

print("\n✓ Saved to: models/learning_curves.png")

## 11. Threshold Optimization

In [ ]:
print("=" * 60)
print("THRESHOLD OPTIMIZATION")
print("=" * 60)

y_val_proba = final_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, y_val_proba)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-10)

optimal_idx = np.argmax(f1_scores[:-1])
optimal_threshold = thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

y_val_pred_default = (y_val_proba >= 0.5).astype(int)
default_f1 = f1_score(y_val, y_val_pred_default)

print(f"\n📊 Threshold Comparison:")
print(f"\n   Default (0.5): F1 = {default_f1:.4f}")
print(f"   Optimal ({optimal_threshold:.3f}): F1 = {optimal_f1:.4f}")
print(f"   Improvement: {optimal_f1 - default_f1:+.4f}")

print("\n📝 Using default threshold (0.5) for consistency")
print("=" * 60)

In [ ]:
# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(recalls, precisions, linewidth=2)
axes[0].scatter([recalls[optimal_idx]], [precisions[optimal_idx]], 
               s=100, c='red', zorder=5, label=f'Optimal')
axes[0].set_xlabel('Recall')
axes[0].set_ylabel('Precision')
axes[0].set_title('Precision-Recall Curve', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(thresholds, f1_scores[:-1], linewidth=2)
axes[1].axvline(optimal_threshold, color='red', linestyle='--', label='Optimal')
axes[1].axvline(0.5, color='orange', linestyle='--', label='Default')
axes[1].set_xlabel('Threshold')
axes[1].set_ylabel('F1-Score')
axes[1].set_title('F1 vs Threshold', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'threshold_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved to: models/threshold_optimization.png")

## 12. Final Test Set Evaluation

**⚠️ Test set touched for the FIRST AND ONLY time!**

In [ ]:
print("\n" + "=" * 70)
print("FINAL EVALUATION ON HELD-OUT TEST SET")
print("=" * 70)

print("\n⚠️  Test set touched for FIRST AND ONLY time!\n")

# Predictions
y_test_pred = final_model.predict(X_test)
y_test_proba = final_model.predict_proba(X_test)[:, 1]

# Metrics
test_metrics = {
    'accuracy': accuracy_score(y_test, y_test_pred),
    'precision': precision_score(y_test, y_test_pred),
    'recall': recall_score(y_test, y_test_pred),
    'f1_score': f1_score(y_test, y_test_pred),
    'roc_auc': roc_auc_score(y_test, y_test_proba)
}

print("=" * 70)
print("FINAL TEST SET PERFORMANCE")
print("=" * 70)
for metric, value in test_metrics.items():
    print(f"{metric.upper():12} : {value:.4f} ({value*100:.2f}%)")
print("=" * 70)

# Confusion matrix
cm = confusion_matrix(y_test, y_test_pred)
tn, fp, fn, tp = cm.ravel()

print("\n📊 Confusion Matrix:")
print(f"\n                Predicted")
print(f"                Benign    Attack")
print(f"   Actual Benign  {tn:>6,}    {fp:>6,}")
print(f"   Actual Attack  {fn:>6,}    {tp:>6,}")

print(f"\n   FN Rate: {fn/(fn+tp)*100:.2f}% (attacks missed)")
print(f"   FP Rate: {fp/(fp+tn)*100:.2f}% (false alarms)")
print("\n" + "=" * 70)

In [ ]:
# Comprehensive visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt=',d', cmap='Blues', ax=axes[0, 0],
            xticklabels=['Benign', 'Attack'],
            yticklabels=['Benign', 'Attack'])
axes[0, 0].set_title('Confusion Matrix', fontweight='bold')
axes[0, 0].set_ylabel('True Label')
axes[0, 0].set_xlabel('Predicted Label')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_test_proba)
axes[0, 1].plot(fpr, tpr, linewidth=2, label=f'AUC = {test_metrics["roc_auc"]:.4f}')
axes[0, 1].plot([0, 1], [0, 1], 'k--', linewidth=1)
axes[0, 1].set_xlabel('False Positive Rate')
axes[0, 1].set_ylabel('True Positive Rate')
axes[0, 1].set_title('ROC Curve', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Probability Distribution
axes[1, 0].hist(y_test_proba[y_test == 0], bins=50, alpha=0.6, 
               label='Benign', color='green')
axes[1, 0].hist(y_test_proba[y_test == 1], bins=50, alpha=0.6, 
               label='Attack', color='red')
axes[1, 0].axvline(0.5, color='black', linestyle='--', linewidth=2)
axes[1, 0].set_xlabel('Predicted Probability')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Probability Distribution', fontweight='bold')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3, axis='y')

# Metrics Bar Chart
metrics_display = ['Accuracy', 'Precision', 'Recall', 'F1-Score', 'ROC-AUC']
metrics_values = [test_metrics['accuracy'], test_metrics['precision'], 
                 test_metrics['recall'], test_metrics['f1_score'], 
                 test_metrics['roc_auc']]
bars = axes[1, 1].bar(metrics_display, metrics_values, alpha=0.7)
axes[1, 1].set_ylabel('Score')
axes[1, 1].set_title('Test Performance', fontweight='bold')
axes[1, 1].set_ylim([0.75, 1.0])
axes[1, 1].grid(True, alpha=0.3, axis='y')
axes[1, 1].tick_params(axis='x', rotation=45)

for bar, value in zip(bars, metrics_values):
    axes[1, 1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01,
                   f'{value:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.savefig(MODEL_DIR / 'test_evaluation.png', dpi=150, bbox_inches='tight')
plt.show()

print("✓ Saved to: models/test_evaluation.png")

## 13. Save Everything to Google Drive

In [ ]:
print("=" * 60)
print("SAVING MODEL AND RESULTS")
print("=" * 60)

# Save model
model_path = MODEL_DIR / "xgboost_optimized.json"
final_model.save_model(str(model_path))
print(f"\n✓ Model saved: {model_path}")

# Save results
results_dict = {
    'model_info': {
        'algorithm': 'XGBoost',
        'version': xgb.__version__,
        'trained_on': 'Google Colab',
        'random_seed': RANDOM_SEED
    },
    'best_hyperparameters': random_search.best_params_,
    'cross_validation': {
        'scores': {k: {'mean': float(v.mean()), 'std': float(v.std())} 
                  for k, v in cv_scores.items()}
    },
    'validation_performance': optimized_metrics,
    'test_performance': test_metrics,
    'confusion_matrix': {
        'tn': int(tn), 'fp': int(fp),
        'fn': int(fn), 'tp': int(tp)
    },
    'dataset_info': {
        'train_samples': int(X_train.shape[0]),
        'val_samples': int(X_val.shape[0]),
        'test_samples': int(X_test.shape[0]),
        'num_features': int(X_train.shape[1])
    }
}

with open(MODEL_DIR / 'xgboost_results.json', 'w') as f:
    json.dump(results_dict, f, indent=2)
print(f"✓ Results saved: {MODEL_DIR / 'xgboost_results.json'}")

# Save predictions
predictions_df = pd.DataFrame({
    'y_true': y_test,
    'y_pred': y_test_pred,
    'y_proba': y_test_proba
})
predictions_df.to_csv(MODEL_DIR / 'xgboost_predictions.csv', index=False)
print(f"✓ Predictions saved: {MODEL_DIR / 'xgboost_predictions.csv'}")

print("\n✅ All artifacts saved to Google Drive!")
print(f"\n📁 Location: {MODEL_DIR}")
print("   Files:")
print("   • xgboost_optimized.json")
print("   • xgboost_results.json")
print("   • xgboost_predictions.csv")
print("   • feature_importance.csv")
print("   • *.png (6 visualizations)")
print("\n💾 Download these files from Google Drive when done!")
print("=" * 60)

## 14. Download Results (Optional)

In [ ]:
# Uncomment to download files directly from Colab
# from google.colab import files

# Download model
# files.download(str(MODEL_DIR / 'xgboost_optimized.json'))

# Download results
# files.download(str(MODEL_DIR / 'xgboost_results.json'))

# Or create a zip of everything
# import shutil
# shutil.make_archive('/content/xgboost_outputs', 'zip', MODEL_DIR)
# files.download('/content/xgboost_outputs.zip')

print("💡 Tip: Files are already saved to Google Drive")
print("   Access them anytime at: drive.google.com")

## 15. Summary

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT COMPLETE!")
print("=" * 70)

print("\n✅ COMPLETED STEPS:")
steps = [
    "Mounted Google Drive",
    "Loaded preprocessed data",
    "Created train-validation split",
    "Trained baseline model",
    "Optimized hyperparameters (50 iterations)",
    "Cross-validation (5-fold)",
    "Feature importance analysis",
    "Learning curves (overfitting check)",
    "Threshold optimization",
    "Final test set evaluation",
    "Saved all results to Google Drive"
]
for i, step in enumerate(steps, 1):
    print(f"   {i:2d}. ✓ {step}")

print("\n" + "=" * 70)
print("FINAL RESULTS")
print("=" * 70)
print(f"   Accuracy:  {test_metrics['accuracy']*100:>6.2f}%")
print(f"   Precision: {test_metrics['precision']*100:>6.2f}%")
print(f"   Recall:    {test_metrics['recall']*100:>6.2f}%")
print(f"   F1-Score:  {test_metrics['f1_score']*100:>6.2f}%")
print(f"   ROC-AUC:   {test_metrics['roc_auc']:>6.4f}")

print("\n" + "=" * 70)
print("NEXT STEPS")
print("=" * 70)
print("   1. Download results from Google Drive")
print("   2. Implement CNN (automatic feature learning)")
print("   3. Implement LSTM (temporal modeling)")
print("   4. Compare all models")
print("   5. Deploy best model")

print("\n" + "=" * 70)
print("✨ XGBoost BASELINE ESTABLISHED! ✨")
print("=" * 70 + "\n")